# Superposition and Polysemanticity in Neural Networks

**Prerequisites**: Notebook 01 (Transformer Circuits)

Neural networks cram more features into their activations than they have dimensions. This is **superposition**, and it's the central obstacle to interpretability. This notebook explores why individual neurons don't map to single features, and how models pull off this trick.

## Section 1: The Problem -- Features vs Neurons

In an ideal world, each neuron would correspond to one interpretable feature. In practice, individual neurons are **polysemantic** -- they activate for multiple unrelated concepts. Why? Because models need to represent far more features than they have dimensions.

**Superposition** is how they do it: encode features as directions in activation space rather than individual neurons. Features can "share" neurons as long as they rarely co-occur.

Here's the key intuition: if you have $d$ neurons but need to represent $n \gg d$ features, you can use directions in $\mathbb{R}^d$ (of which there are infinitely many). The constraint is interference -- features sharing dimensions will interfere with each other.

### Information-Theoretic View of Superposition

The toy model results in subsequent sections can be understood through a precise mathematical lens. The fundamental question is: **how many nearly-orthogonal vectors can be packed into $\mathbb{R}^d$?**

**Johnson-Lindenstrauss perspective.** The JL lemma tells us that $n$ points in high-dimensional space can be projected to $d = O(\varepsilon^{-2} \log n)$ dimensions while preserving all pairwise distances to within factor $(1 \pm \varepsilon)$. Inverting this: in $d$ dimensions, we can find $n \sim e^{O(\varepsilon^2 d)}$ vectors that are $\varepsilon$-nearly-orthogonal (i.e., $|\langle f_i, f_j \rangle| \leq \varepsilon$ for all $i \neq j$). This is **exponentially many** vectors — far more than the $d$ exactly orthogonal ones.

**The interference cost model.** Suppose we represent $n$ unit-norm features $\{f_i\}_{i=1}^n$ in $\mathbb{R}^d$. When feature $i$ is active with value $a_i$, the reconstruction of feature $j$ suffers interference:

$$\hat{x}_j = a_j + \sum_{i \neq j} a_i \langle f_i, f_j \rangle$$

The squared reconstruction error on feature $j$ due to feature $i$ is $a_i^2 \langle f_i, f_j \rangle^2$. In expectation:

$$\mathbb{E}[\text{interference from } i \text{ on } j] = \mathbb{E}[a_i^2] \cdot \langle f_i, f_j \rangle^2 \cdot \Pr[i \text{ active}]$$

**The sparsity-interference tradeoff.** Let $S$ denote the sparsity (probability of a feature being *inactive*). Then $\Pr[\text{both } i \text{ and } j \text{ active}] = (1-S)^2$. The total expected interference cost for feature $j$ is:

$$C_j = \sum_{i \neq j} \text{importance}_i \cdot (1-S)^2 \cdot \langle f_i, f_j \rangle^2$$

The benefit of representing feature $j$ at all (vs. ignoring it) is $\text{importance}_j \cdot (1-S)$ (the reduction in expected loss from being able to reconstruct it when active).

**Phase transition criterion.** Feature $k$ should be placed in superposition when:

$$\text{importance}_k \cdot (1-S) > \sum_{j \neq k} \text{importance}_j \cdot (1-S)^2 \cdot |\langle f_k, f_j \rangle|^2$$

Dividing both sides by $(1-S)$:

$$\text{importance}_k > (1-S) \cdot \sum_{j \neq k} \text{importance}_j \cdot |\langle f_k, f_j \rangle|^2$$

This makes the role of sparsity transparent: as $S \to 1$, the RHS vanishes and *any* feature is worth representing, regardless of interference. As $S \to 0$ (dense features), the interference cost dominates and only $d$ features survive.

**Connection to the toy model.** The phase transitions we observe in Section 5 correspond exactly to the points where this inequality flips for successive features (ordered by importance). At sparsity $S^*_k$, feature $k$ transitions from "not worth representing" to "worth putting in superposition." The geometric decrease in importance ($0.7^i$) means features transition in order from most to least important.

## Section 2: Toy Model of Superposition

Following [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) (Elhage et al., 2022), we'll build a toy model: an autoencoder that takes $n$-dimensional sparse input, projects to $d < n$ dimensions, then reconstructs. We'll study when and how the model puts features in superposition.

**Model**: $\hat{x} = \text{ReLU}(W^T W x + b)$ where $W$ is $d \times n$ ($d < n$)
- Input $x$ has $n$ features, each active with probability $1 - S$ (sparsity $S$)
- Features have varying importance (weight)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

class ToyModel(nn.Module):
    """Toy model of superposition: n features -> d dimensions -> n reconstructed features."""
    def __init__(self, n_features, d_hidden):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_hidden, n_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(n_features))
    
    def forward(self, x):
        # Encode: x @ W^T -> hidden (d_hidden dims)
        # Decode: hidden @ W -> reconstruction (n_features dims)
        hidden = x @ self.W.T  # (batch, d_hidden)
        reconstructed = torch.relu(hidden @ self.W + self.b)  # (batch, n_features)
        return reconstructed

def generate_data(batch_size, n_features, sparsity, importance):
    """Generate sparse input data with varying feature importance."""
    # Each feature is active with probability (1 - sparsity)
    mask = (torch.rand(batch_size, n_features, device=device) > sparsity).float()
    values = torch.rand(batch_size, n_features, device=device)
    return mask * values, importance

def train_model(n_features, d_hidden, sparsity, n_steps=10000, lr=1e-3):
    """Train toy model with given sparsity level."""
    # Feature importance decreases geometrically
    importance = torch.tensor([0.7 ** i for i in range(n_features)], device=device)
    
    model = ToyModel(n_features, d_hidden).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    for step in range(n_steps):
        x, imp = generate_data(256, n_features, sparsity, importance)
        x_hat = model(x)
        # Weighted MSE loss (more important features matter more)
        loss = ((x - x_hat) ** 2 * imp).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if step % 1000 == 0:
            losses.append(loss.item())
    
    return model, losses

# Train with different sparsity levels
n_features = 20
d_hidden = 5
sparsity_levels = [0.0, 0.9, 0.99, 0.999]

models = {}
for s in sparsity_levels:
    print(f"Training with sparsity={s}...")
    models[s], losses = train_model(n_features, d_hidden, s)
    print(f"  Final loss: {losses[-1]:.4f}")

In [ ]:
# Napkin-sized superposition: 5 features crammed into 2 dimensions
# This is small enough to draw by hand and see the geometry directly.

napkin_model, _ = train_model(n_features=5, d_hidden=2, sparsity=0.99, n_steps=15000)

W = napkin_model.W.detach().cpu().numpy()  # shape: (2, 5)
n_feat = W.shape[1]

# Compute column norms (how well each feature is represented)
col_norms = np.linalg.norm(W, axis=0)

# Normalize columns for direction plotting
W_norm = W / (col_norms[np.newaxis, :] + 1e-8)

fig, ax = plt.subplots(figsize=(8, 8))

# Draw unit circle for reference
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), 'k-', alpha=0.15, linewidth=1)

# Plot each feature as an arrow from the origin
threshold = 0.5
for i in range(n_feat):
    represented = col_norms[i] > threshold
    color = plt.cm.tab10(i) if represented else (0.7, 0.7, 0.7, 0.6)
    label = f"f{i} (norm={col_norms[i]:.2f})" if represented else f"f{i} (lost, norm={col_norms[i]:.2f})"

    # Draw arrow showing the actual (unnormalized) feature vector
    ax.annotate("", xy=(W[0, i], W[1, i]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", lw=2.5 if represented else 1.2,
                                color=color, mutation_scale=18))

    # Label at the tip of the normalized direction (slightly outside)
    offset = 1.15
    ax.text(W_norm[0, i] * offset, W_norm[1, i] * offset, label,
            fontsize=9, ha='center', va='center', color=color,
            fontweight='bold' if represented else 'normal')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.3)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.3)
ax.grid(True, alpha=0.2)
ax.set_xlabel("Hidden dim 1")
ax.set_ylabel("Hidden dim 2")
ax.set_title(f"Superposition in 2D: {n_feat} features packed into 2 dimensions\n"
             f"(sparsity=0.99, represented = norm > {threshold})")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=plt.cm.tab10(0), lw=2.5, label=f"Represented (norm > {threshold})"),
    Line2D([0], [0], color=(0.7, 0.7, 0.7), lw=1.2, label=f"Lost (norm <= {threshold})"),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

# Print pairwise angles between represented features
print("Pairwise angles between represented features:")
represented_idx = [i for i in range(n_feat) if col_norms[i] > threshold]
for i, fi in enumerate(represented_idx):
    for fj in represented_idx[i+1:]:
        cos_sim = np.dot(W_norm[:, fi], W_norm[:, fj])
        angle = np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))
        print(f"  f{fi} <-> f{fj}: {angle:.1f} degrees (cos_sim={cos_sim:.3f})")

## Section 3: Visualizing Superposition

We can visualize what the model learned by examining $W^T W$. The diagonal shows how well each feature is represented (1.0 = perfectly represented). Off-diagonal entries show interference between features.

- At **low sparsity** (features often co-occur), the model can only represent $d$ features and ignores the rest.
- At **high sparsity** (features rarely co-occur), the model can put MORE than $d$ features into superposition, accepting some interference because features rarely collide.

In [ ]:
fig, axes = plt.subplots(1, len(sparsity_levels), figsize=(5 * len(sparsity_levels), 5))

for idx, s in enumerate(sparsity_levels):
    W = models[s].W.detach().cpu()
    WtW = (W.T @ W).numpy()
    
    ax = axes[idx]
    im = ax.imshow(WtW, cmap="RdBu", vmin=-1, vmax=1)
    ax.set_title(f"W^T W (sparsity={s})")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Feature")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Feature Representation at Different Sparsity Levels", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.5
print(f"Bottleneck dimension: {d_hidden}")
print(f"Total features: {n_features}")
print()
for s in sparsity_levels:
    W = models[s].W.detach().cpu()
    WtW = W.T @ W
    diag = WtW.diag().numpy()
    n_represented = (diag > threshold).sum()
    print(f"Sparsity {s}: {n_represented}/{n_features} features represented (diagonal > {threshold})")
    # Show diagonal values
    print(f"  Diagonal: {np.round(diag[:10], 2)} ...")

## Section 4: Geometry of Superposition

How do features arrange themselves geometrically when in superposition? In 2D, the optimal arrangement is related to regular polytopes. For example, 3 features in 2D form the vertices of an equilateral triangle (120 degrees apart). More features push toward uniform distributions on the sphere.

The key insight: features in superposition are **almost orthogonal** directions in activation space. They interfere only slightly, and if features are sparse enough, that interference rarely matters.

In [ ]:
# Train a 2D bottleneck model with many features at high sparsity
model_2d, _ = train_model(n_features=8, d_hidden=2, sparsity=0.99, n_steps=20000)

W = model_2d.W.detach().cpu().numpy()  # (2, 8)

# Normalize feature vectors
norms = np.linalg.norm(W, axis=0, keepdims=True)
W_norm = W / (norms + 1e-8)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Plot unit circle
theta = np.linspace(0, 2 * np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k-', alpha=0.2)

# Plot feature directions
colors = cm.viridis(np.linspace(0, 1, W.shape[1]))
for i in range(W.shape[1]):
    norm = norms[0, i]
    if norm > 0.1:  # Only plot represented features
        ax.arrow(0, 0, W_norm[0, i] * 0.9, W_norm[1, i] * 0.9,
                head_width=0.05, head_length=0.05, fc=colors[i], ec=colors[i])
        ax.annotate(f"f{i}", (W_norm[0, i] * 1.1, W_norm[1, i] * 1.1), fontsize=10, color=colors[i])

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title(f"Feature Directions in 2D (8 features, sparsity=0.99)\nNorms: {np.round(norms[0], 2)}")
plt.show()

### Polytope Structure and Optimal Packings

The geometry of superposition connects to classical problems in combinatorics and coding theory. The feature directions $\{f_i\}$ in $\mathbb{R}^d$ must be arranged to minimize worst-case interference — this is equivalent to packing points on the unit sphere $S^{d-1}$.

**$d = 2$: Exact solutions.** On the unit circle $S^1$, the optimal packing of $k$ points that minimizes maximum $|\langle f_i, f_j \rangle|$ is the regular $k$-gon:

| Features $k$ | Arrangement | Max $|\langle f_i, f_j \rangle|$ | Geometry |
|:---:|:---:|:---:|:---|
| 2 | Orthogonal pair | 0 | Two perpendicular vectors |
| 3 | Equilateral triangle | $\cos(60°) = 0.5$ | Mercedes-Benz frame |
| 4 | Square | $\cos(45°) \approx 0.707$ | Two orthogonal pairs |
| 5 | Regular pentagon | $\cos(36°) \approx 0.809$ | |
| 6 | Regular hexagon | $\cos(30°) \approx 0.866$ | Three antipodal pairs |

Note the jump at $k = 3$: adding one feature beyond the $d = 2$ orthogonal basis incurs interference of 0.5. Whether this is worthwhile depends entirely on the sparsity-importance tradeoff derived above.

**The antipodal structure.** In the toy model and in real networks, features can use **both directions** of a vector: $+f_i$ and $-f_i$ encode different features (since ReLU or similar activation functions distinguish positive from negative projections). This effectively doubles the number of usable directions. In $d = 2$, three features arranged as an equilateral triangle give six usable directions (three pairs of antipodal features). This is why the 2D plot above often shows features at $180°$ — they are antipodal pairs.

**Higher dimensions: Tammes problem.** In $\mathbb{R}^d$, the optimal packing of $n$ points on $S^{d-1}$ that maximizes the minimum angle between any two points is known as the **Tammes problem** (or the generalized Thomson problem). Exact solutions are known only for small $n$ and $d$. For large $n \gg d$, results from coding theory give bounds:
- **Welch bound**: For any set of $n$ unit vectors in $\mathbb{R}^d$, $\max_{i \neq j} |\langle f_i, f_j \rangle| \geq \sqrt{\frac{n - d}{d(n-1)}}$. Equality defines an **equiangular tight frame** (ETF).
- ETFs exist only for $n \leq d^2$ (in $\mathbb{R}^d$ over reals), giving a theoretical maximum superposition capacity.

**Connection to real models.** Empirical analysis of trained SAE decoder columns (which represent learned feature directions) shows:
1. Angular separation between features is significantly better than random — models learn near-optimal packings.
2. Feature pairs with high cosine similarity tend to be semantically related or rarely co-occurring (validating the sparsity-interference theory).
3. The distribution of pairwise cosine similarities concentrates near zero, consistent with approximate equiangularity — exactly what the theory predicts for near-optimal superposition.

## Section 5: Phase Transitions

As sparsity increases, the model undergoes **phase transitions** — sudden changes in how many features it represents. At a critical sparsity level, it becomes worth putting an additional feature in superposition. This is analogous to phase transitions in physics.

The transition happens when:

$$\text{benefit of representing feature} > \text{cost of interference} \times \text{probability of co-occurrence}$$

In [ ]:
sparsity_range = np.linspace(0.0, 0.999, 20)
n_represented_list = []

for s in sparsity_range:
    m, _ = train_model(n_features=20, d_hidden=5, sparsity=s, n_steps=5000)
    W = m.W.detach().cpu()
    diag = (W.T @ W).diag().numpy()
    n_represented_list.append((diag > 0.3).sum())

plt.figure(figsize=(10, 5))
plt.plot(sparsity_range, n_represented_list, 'bo-')
plt.axhline(y=5, color='r', linestyle='--', label='Bottleneck dim (d=5)')
plt.xlabel("Sparsity (fraction of zeros)")
plt.ylabel("Number of represented features")
plt.title("Phase Transitions: Features Represented vs Sparsity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
### Running Example: IOI — Why Superposition Matters

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

**Why superposition is essential for IOI**: The IOI circuit involves approximately 26 attention heads spanning most of the model's 12 layers. These heads play distinct functional roles — name mover heads, S-inhibition heads, duplicate token heads, and previous token heads — each operating on task-specific features.

If each head's computation used only dedicated, non-superposed features, the model would need enormous capacity to support this single task alongside the thousands of other linguistic behaviors it handles. Superposition lets the model pack many task-specific circuits into shared neural substrate. The name mover heads, S-inhibition heads, and duplicate token heads all operate on features that exist in superposition — they share dimensions with features used for completely unrelated tasks.

This is why studying superposition (this notebook) is essential context for understanding circuits like IOI (notebook 04): the circuit's components are not isolated — they are interleaved with other circuits via superposition, and the features they manipulate are directions in activation space, not dedicated neurons.

---
## Exercises

### Exercise 1: Vary Sparsity in the Toy Model

Modify the toy model to test different sparsity levels (S = 0.01, 0.05, 0.1, 0.3, 0.7). For each, train the model and measure how many features are represented (using the W^T W diagonal). Plot the number of "active" features vs. sparsity level.

This will let you observe the phase transition between "no superposition" (only d features represented) and "full superposition" (many more than d features represented).

<details>
<summary>Hint</summary>

A feature is "active" (represented) if its column in W has norm > 0.5, which corresponds to the diagonal of W^T W being > 0.25. Use `torch.norm(W, dim=0)` to get column norms. The `train_model` function defined earlier in this notebook accepts a `sparsity` parameter -- call it in a loop over your sparsity levels.

</details>

In [ ]:
sparsity_levels = [0.01, 0.05, 0.1, 0.3, 0.7]
n_features = 20
d_hidden = 5
active_counts = []

# TODO: Try different sparsity levels or change the norm threshold!
for s in sparsity_levels:
    m, _ = train_model(n_features, d_hidden, s, n_steps=10000)
    W = m.W.detach().cpu()
    col_norms = torch.norm(W, dim=0)
    n_active = (col_norms > 0.5).sum().item()
    active_counts.append(n_active)
    print(f"Sparsity {s}: {n_active}/{n_features} features active")

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(sparsity_levels, active_counts, 'bo-', markersize=8)
plt.axhline(y=d_hidden, color='r', linestyle='--', label=f'Bottleneck dim (d={d_hidden})')
plt.xlabel("Sparsity (probability of being inactive)")
plt.ylabel("Number of active features (||w_i|| > 0.5)")
plt.title("Features Represented vs. Sparsity Level")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 2: Geometric Analysis of Learned Features

For a trained toy model with n_features=8 and d_hidden=3, compute all pairwise angles between feature directions. Are they close to known optimal packings? Plot a histogram of pairwise cosine similarities (off-diagonal entries of the Gram matrix).

For 8 vectors in R^3, the optimal packing (minimizing maximum inner product) should have maximum inner product around 0.33. Compare your trained model's geometry to this theoretical optimum.

<details>
<summary>Hint</summary>

Train with `train_model(n_features=8, d_hidden=3, sparsity=0.99, n_steps=20000)`. Normalize columns of W: `W_norm = W / W.norm(dim=0, keepdim=True)`. Compute the Gram matrix: `G = W_norm.T @ W_norm`. The off-diagonal entries are the pairwise cosine similarities. Extract them with a mask: `off_diag = G[~torch.eye(8, dtype=bool)]`. Plot with `plt.hist()`.

</details>

In [ ]:
# Exercise 2: Geometric Analysis of Learned Features

# 1. Train a toy model with 8 features in 3 dimensions at high sparsity
# TODO: Try different n_features, d_hidden, or sparsity values!
model_3d, _ = train_model(n_features=8, d_hidden=3, sparsity=0.99, n_steps=20000)

# 2. Get the trained W matrix and normalize columns
W = model_3d.W.detach().cpu()
W_norm = W / W.norm(dim=0, keepdim=True)

# 3. Compute the Gram matrix (pairwise cosine similarities)
G = W_norm.T @ W_norm

# 4. Extract off-diagonal entries
mask = ~torch.eye(8, dtype=bool)
off_diag = G[mask].numpy()

# 5. Plot histogram of pairwise cosine similarities
plt.figure(figsize=(10, 5))
plt.hist(off_diag, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(x=0.33, color='r', linestyle='--', label='Optimal max inner product (~0.33)')
plt.axvline(x=-0.33, color='r', linestyle='--')
plt.xlabel("Cosine Similarity")
plt.ylabel("Count")
plt.title("Pairwise Cosine Similarities of Learned Feature Directions (8 features in R^3)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 6. Print summary statistics
print(f"Max |cosine similarity|: {np.abs(off_diag).max():.3f}")
print(f"Mean |cosine similarity|: {np.abs(off_diag).mean():.3f}")
print(f"Theoretical optimal max inner product for 8 vectors in R^3: ~0.33")

## Section 6: Implications for Interpretability

Superposition has profound consequences:

1. **Individual neurons are not the right unit of analysis.** A neuron's activation is a mixture of multiple features in superposition. Interpreting individual neurons is misleading.

2. **Features are directions, not neurons.** The correct unit is a direction in activation space, which may be a linear combination of many neurons.

3. **This motivates Sparse Autoencoders.** If features are directions, we need a method to discover them. SAEs (next notebook) learn to decompose activations into sparse combinations of learned feature directions.

4. **The number of features scales with sparsity.** Sparser, more specialized features can be packed more densely. This explains why LLMs can represent enormous numbers of concepts.

**Key open questions:**
- Can we recover ALL features from superposition?
- How do features interact when they do co-occur?
- Does superposition differ qualitatively between small and large models?

**Next**: Notebook 03 -- Sparse Autoencoders (the practical tool for extracting features from superposition)

**References:**
- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) (Elhage et al., 2022)
- [Superposition, Memorization, and Double Descent](https://transformer-circuits.pub/2023/toy-double-descent/index.html) (Anthropic, 2023)